## Inisialisasi dan Import Library

1. pandas (pd): Digunakan untuk membaca, memanipulasi, dan menstrukturkan data dari file CSV ke dalam bentuk DataFrame.

2. numpy (np): Digunakan untuk melakukan komputasi numerik numerik dasar, seperti menghitung nilai rata-rata (mean) dan standar deviasi sampel.

3. sys & os: Pustaka bawaan Python untuk berinteraksi dengan sistem operasi, berguna dalam manajemen pencarian jalur folder proyek secara dinamis.

In [22]:
import sys
import os
import pandas as pd
import numpy as np

root_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

print("Jalur proyek yang SEBENARNYA terdeteksi:", root_path)

Jalur proyek yang SEBENARNYA terdeteksi: d:\Tugas Statistika\stat-audit-pandas-sti-2025-main


## Pendefinisian Fungsi Komputasi Uji-Z Dua Sampel

Perhitungan di dalam fungsi ini mengikuti landasan teoretis dari literatur statistika, dengan tahapan:
1. Menghitung Pooled Standard Error ($SE$) gabungan dari kedua varians sampel.
2. Menghitung nilai $Z$-statistik berdasarkan selisih rata-rata kedua kelompok sampel.
3. Menghitung $P$-value secara akurat memanfaatkan fungsi distribusi kumulatif normal standar dari scipy.stats.norm.cdf.
4. Mengambil keputusan formal berbasis tingkat signifikansi ($\alpha = 0.05$) tanpa menggunakan frasa bias "Menerima H0", melainkan menggunakan standar akademik: "Reject H0" atau "Fail to reject H0".

In [23]:
import scipy.stats as stats

def z_test_two_sample(x_bar1, x_bar2, sigma1, sigma2, n1, n2, alternative='two-sided', alpha=0.05):
    """
    Menghitung Uji Z Dua Sampel Bebas secara manual berdasarkan Tsun (2020) hal. 309.
    """
    pooled_se = np.sqrt((sigma1**2 / n1) + (sigma2**2 / n2))
    z_stat = (x_bar1 - x_bar2) / pooled_se
    
    if alternative == 'two-sided':
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    elif alternative == 'less':
        p_value = stats.norm.cdf(z_stat)
    elif alternative == 'greater':
        p_value = 1 - stats.norm.cdf(z_stat)
        
    if p_value < alpha:
        decision = "Reject H0"
        interpretation = f"Tolak H0 pada tingkat signifikansi {alpha}. Rata-rata kedua kelompok berbeda secara signifikan."
    else:
        decision = "Fail to reject H0"
        interpretation = f"Gagal menolak H0 pada tingkat signifikansi {alpha}. Tidak ada perbedaan rata-rata yang signifikan."
        
    return {
        "z_stat": z_stat,
        "p_value": p_value,
        "decision": decision,
        "interpretation": interpretation
    }

print("Fungsi z_test_two_sample berhasil dimuat langsung di dalam Notebook!")

Fungsi z_test_two_sample berhasil dimuat langsung di dalam Notebook!


## Pembacaan Dataset Hasil Preprocessing

Fungsi pd.read_csv membaca file dataset.csv menggunakan jalur relatif (relative path) dari folder data/clean/. Output konfirmasi menunjukkan bahwa data berhasil dimuat dengan total volume sebanyak 3.000 baris observasi.

In [24]:
df = pd.read_csv('../data/clean/dataset.csv')

print("Berhasil membaca dataset! Jumlah baris data:", len(df))

Berhasil membaca dataset! Jumlah baris data: 3000


## Pemeriksaan Struktur Kolom Dataset

Tahap sanity check untuk menginspeksi seluruh nama kolom (features) yang tersedia di dalam dataset. Langkah ini krusial untuk memastikan sinkronisasi antara logika kode analitik dengan nama kolom riil yang diberikan oleh Data Engineer. Hasil pengecekan mengonfirmasi bahwa kolom kategori isu bernama 'labels' dan kolom durasi penyelesaian berbentuk numerik bernama 'days_to_close'.

In [25]:
print(df.columns.tolist())

['number', 'title', 'is_pr', 'state', 'created_at', 'closed_at', 'days_to_close', 'labels', 'has_bug', 'has_enhancement', 'merged', 'user_login']


## Partisi Data dan Penanganan Missing Values

Melakukan pemisahan (slicing) dan penyaringan data menjadi dua kelompok sampel independen yang akan diuji:

1. durasi_bug: Durasi waktu (dalam satuan hari) yang dibutuhkan untuk menyelesaikan isu berkategori kerusakan sistem (bug).
2. durasi_enhancement: Durasi waktu (dalam satuan hari) yang dibutuhkan untuk menyelesaikan isu berkategori pengembangan fitur (enhancement).
Fungsi .dropna() diaplikasikan di akhir untuk membuang data kosong (missing values) pada masing-masing kelompok sampel agar tidak merusak validitas perhitungan matematika kalkulus statistik selanjutnya.

In [26]:
durasi_bug = df[df['labels'].str.lower() == 'bug']['days_to_close'].dropna()
durasi_enhancement = df[df['labels'].str.lower() == 'enhancement']['days_to_close'].dropna()

## Eksekusi Pengujian Hipotesis dan Output Analisis Audit

Sistem mengekstrak parameter statistik deskriptif dari masing-masing sampel (jumlah sampel $n$, rata-rata $\bar{x}$, dan standar deviasi sampel $s$ dengan parameter derajat bebas ddof=1).Nilai-nilai tersebut kemudian diumpankan ke dalam fungsi z_test_two_sample yang telah dibuat pada Cell 2. Hasil cetakan akhir menyajikan laporan formal audit statistik proyek open-source pandas-dev/pandas secara komprehensif, mencakup skor statistik uji ($Z = -0.8563$), nilai probabilitas ($P\text{-value} = 0.3918$), keputusan uji, serta interpretasi ilmiah

In [28]:
n_bug = len(durasi_bug)
x_bar_bug = np.mean(durasi_bug)
sigma_bug = np.std(durasi_bug, ddof=1)

n_enhancement = len(durasi_enhancement)
x_bar_enhancement = np.mean(durasi_enhancement)
sigma_enhancement = np.std(durasi_enhancement, ddof=1)

hasil = z_test_two_sample(
    x_bar1=x_bar_bug, x_bar2=x_bar_enhancement,
    sigma1=sigma_bug, sigma2=sigma_enhancement,
    n1=n_bug, n2=n_enhancement,
    alternative='two-sided',
    alpha=0.05
)

print("="*50)
print("AUDIT STATISTIK: pandas-dev/pandas")
print("="*50)
print(f"Sampel Isu 'bug'        : n = {n_bug}, Rata-rata = {x_bar_bug:.2f} hari, StDev = {sigma_bug:.2f}")
print(f"Sampel Isu 'enhancement': n = {n_enhancement}, Rata-rata = {x_bar_enhancement:.2f} hari, StDev = {sigma_enhancement:.2f}")
print("-"*50)
print(f"Z-Statistik : {hasil['z_stat']:.4f}")
print(f"P-Value     : {hasil['p_value']:.4f}")
print(f"Keputusan   : {hasil['decision']}")
print(f"Interpretasi: {hasil['interpretation']}")
print("="*50)

AUDIT STATISTIK: pandas-dev/pandas
Sampel Isu 'bug'        : n = 52, Rata-rata = 5.29 hari, StDev = 6.82
Sampel Isu 'enhancement': n = 6, Rata-rata = 11.17 hari, StDev = 16.65
--------------------------------------------------
Z-Statistik : -0.8563
P-Value     : 0.3918
Keputusan   : Fail to reject H0
Interpretasi: Gagal menolak H0 pada tingkat signifikansi 0.05. Tidak ada perbedaan rata-rata yang signifikan.
